In [ ]:
# imbalanced-learn provides SMOTE and the imblearn Pipeline
%pip install imbalanced-learn

AI Prompt Audit 

Unedited AI code raw cell, not run

Audit

Is SMOTE where it should be? Yes. The AI put SMOTE inside imblearn.pipeline.Pipeline, between the ColumnTransformer and the DecisionTreeClassifier. GridSearchCV refits the whole pipeline on each training fold, so SMOTE only resamples training data and the validation folds are never touched. It did not apply SMOTE globally before cross-validation, so there is no data leakage.

Bug 1. The data loading does not work for Adult. The file has no header row and marks missing values with ' ?'. A plain pd.read_csv uses the first person's row as the column names and keeps '?' as a normal category, so the imputers never see a missing value. Fix: load_data() reads the file with names=COLUMNS, skipinitialspace=True, and na_values="?".

Bug 2. scoring="f1" crashes. Adult's labels are the strings '>50K' and '<=50K', but the F1 scorer expects the positive class to be 1, so GridSearchCV raises ValueError: pos_label=1 is not a valid label. Fix: load_data() encodes >50K as 1 and <=50K as 0.

In [ ]:
# final debugged pipeline: the AI's code with only the data loading fixed
import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

# -------------------------------------------------
# Load dataset
# -------------------------------------------------
# FIX for bug 1 and bug 2: load_data() names the columns, turns ? into NaN, and encodes >50K as 1

COLUMNS = ["age", "workclass", "fnlwgt", "education", "education-num",
           "marital-status", "occupation", "relationship", "race", "sex",
           "capital-gain", "capital-loss", "hours-per-week", "native-country",
           "income"]

def load_data(path="data/adult.data") -> tuple[pd.DataFrame, pd.Series]:

    df = pd.read_csv(path, names=COLUMNS, header=None,
                     skipinitialspace=True, na_values="?")

    # we encode the target as 1 for >50K and 0 for <=50K
    y = (df["income"].str.rstrip(".") == ">50K").astype(int)

    # we split off the features by dropping the target and any columns we chose to drop
    X = df.drop(columns=["income"])

    return X, y

X, y = load_data()

# -------------------------------------------------
# Identify feature types
# -------------------------------------------------
numeric_features = X.select_dtypes(include=["int64", "float64"]).columns
categorical_features = X.select_dtypes(include=["object", "category"]).columns

# -------------------------------------------------
# Preprocessing
# -------------------------------------------------
numeric_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features)
    ]
)

# -------------------------------------------------
# Train/Test Split
# -------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

# -------------------------------------------------
# Pipeline
# -------------------------------------------------
pipeline = ImbPipeline([
    ("preprocessor", preprocessor),
    ("smote", SMOTE(random_state=42)),
    ("classifier", DecisionTreeClassifier(random_state=42))
])

# -------------------------------------------------
# Hyperparameter Grid
# -------------------------------------------------
param_grid = {
    "classifier__criterion": ["gini", "entropy"],
    "classifier__max_depth": [None, 5, 10, 20],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__min_samples_leaf": [1, 2, 4]
}

# -------------------------------------------------
# Grid Search
# -------------------------------------------------
grid_search = GridSearchCV(
    estimator=pipeline,
    param_grid=param_grid,
    scoring="f1",
    cv=5,
    n_jobs=-1,
    verbose=1
)

# Train
grid_search.fit(X_train, y_train)

# -------------------------------------------------
# Results
# -------------------------------------------------
print("Best Parameters:")
print(grid_search.best_params_)

print("\nBest Cross-Validation F1 Score:")
print(grid_search.best_score_)

# -------------------------------------------------
# Evaluate
# -------------------------------------------------
y_pred = grid_search.predict(X_test)

print("\nClassification Report:")
print(classification_report(y_test, y_pred))